# ROS2 Humble 机器人·实机部署清单
**平台**：四轮小车（阿克曼 / 麦轮） · Jetson Orin NX 8G · ROS2 Humble

> 状态图例：✅ 已完成 · ⬜ 待完成 · 🔄 已替换算法

## 零、工程架构地图（先看这里再往下读）

> 📌 本节把 `src/` 下所有代码的**调用关系**拆成 5 张图逐层讲清楚。
> 先看全局鸟瞰，再逐层放大到具体文件。

---

### 0.1 全局鸟瞰：src 下的 ROS2 包分层

```
┌─────────────────────────────────────────────────────────────────┐
│                     你要启动的两个入口                            │
│   slam.launch.py（建图模式）    navigation.launch.py（导航模式）   │
└──────────┬──────────────────────────────┬───────────────────────┘
           │                              │
           ▼                              ▼
┌─────────────────────┐     ┌───────────────────────────────────┐
│    slam 包           │     │    navigation 包                   │
│  SLAM Toolbox 建图   │     │  Nav2 全家桶（AMCL+规划+控制器）    │
│  config/slam.yaml    │     │  config/nav2_params.yaml           │
│                      │     │  config/nav2_controller_dwb.yaml   │
└──────────┬───────────┘     └──────────────┬────────────────────┘
           │  两者都调用                      │
           ▼                                ▼
┌─────────────────────────────────────────────────────────────────┐
│             slam/launch/include/robot.launch.py                  │
│             "基座 launch"——启动底盘 + 传感器 + EKF               │
│                                                                  │
│   ┌──────────────┐  ┌──────────────┐  ┌──────────────────────┐  │
│   │ controller   │  │ peripherals  │  │ rosorin_description  │  │
│   │ 底盘驱动+EKF │  │ 雷达/IMU/相机│  │     URDF模型         │  │
│   └──────┬───────┘  └──────┬───────┘  └──────────┬───────────┘  │
│          │                 │                     │               │
│          ▼                 ▼                     ▼               │
│   ros_robot_controller    硬件驱动节点      .xacro → TF树       │
│   (电机/编码器/IMU 底层)                                         │
└─────────────────────────────────────────────────────────────────┘
```

**一句话理解**：不管你是建图还是导航，都要先把"基座"（底盘 + 传感器 + URDF）跑起来。区别只是上面叠的是 SLAM Toolbox 还是 Nav2。

### 0.2 放大看"基座层"：controller.launch.py 到底启动了什么

> 这是最核心的一层，建图和导航都依赖它。文件在 `src/driver/controller/launch/controller.launch.py`

```
controller.launch.py
│
├─① odom_publisher.launch.py ─────────────────────────────────────
│   │
│   ├─ robot_description.launch.py        ← 加载 URDF xacro → 发布 TF 静态变换
│   │   └─ 读取 rosorin_description/urdf/mecanum.xacro（或 acker.xacro）
│   │       定义了: base_footprint → base_link → lidar_frame
│   │                                          → imu_link
│   │                                          → camera_link0
│   │
│   ├─ ros_robot_controller.launch.py     ← 底层硬件驱动（电机+编码器+IMU）
│   │   发布: /imu/data, 接收: /cmd_vel → 电机
│   │
│   └─ odom_publisher 节点                ← 读编码器 → 发布 /odom_raw + odom→base TF
│       配置: config/calibrate_params.yaml（轮距、轮半径等）
│
├─② imu_filter.launch.py                 ← imu_filter_madgwick（IMU 姿态滤波）
│   输入: /imu/data（裸数据）
│   输出: /imu/data（带 orientation 的融合数据）
│
├─③ ekf_filter_node                      ← robot_localization EKF 融合
│   配置: config/ekf.yaml
│   输入: /odom_raw（轮速） + /imu（IMU） + /odom_rf2o（激光里程计，可选）
│   输出: /odom（融合后里程计） + 发布 odom→base_footprint TF
│
└─④ servo_controller.launch.py           ← 舵机控制（云台等，非导航核心）
```

**关键理解**：
- `/odom_raw` 是原始轮速里程计（会漂移）
- 经过 EKF 融合后变成 `/odom`（稳定版），这个 `/odom` 才是给 SLAM 和 Nav2 用的
- EKF 同时接管了 `odom→base_footprint` 的 TF 发布（所以驱动节点不能重复发布）

### 0.3 放大看"传感器层"：peripherals 包启动了什么

> 文件在 `src/peripherals/launch/`

```
peripherals 包
│
├─ lidar.launch.py ────────────────────────────
│   根据环境变量 LIDAR_TYPE（MS200 / LD19 等）选择对应驱动
│   发布: /scan_raw（原始激光数据）
│   可选: laser_filters 节点 → /scan_raw 滤波后 → /scan
│   配置: config/lidar_filters_config_*.yaml（按型号选）
│
├─ imu_filter.launch.py ──────────────────────
│   启动 imu_filter_madgwick 节点
│   输入: /imu/data（裸加速度+角速度）
│   输出: /imu/data（补上 orientation 四元数）
│
├─ depth_camera.launch.py ────────────────────
│   深度相机驱动（视觉模块用，导航不必须）
│
├─ joystick_control.launch.py ────────────────
│   手柄遥控 → 发布 /cmd_vel（调试建图时用）
│
└─ teleop_key_control.launch.py ──────────────
    键盘遥控 → 发布 /cmd_vel（没手柄时用）
```

**重要环境变量**（在 launch 前必须设置）：
```bash
export LIDAR_TYPE=LD19          # 你的雷达型号
export DEPTH_CAMERA_TYPE=...    # 深度相机型号（可选）
export need_compile=True        # 是否 colcon build 过（True=用 install 路径）
```

### 0.4 话题数据流：从传感器到导航指令的完整链路

> 这张图展示数据怎么从硬件一路流到轮子转动。**箭头 = ROS2 话题**。

```
【硬件层】                    【ROS2 节点】                      【话题】
                                                                
电机编码器 ──→ ros_robot_controller ──→ odom_publisher ──→ /odom_raw
                                                              │
板载 IMU ───→ ros_robot_controller ─────────────────────→ /imu/data（裸）
                                                              │
                      imu_filter_madgwick ←───────────────────┘
                              │
                              └──────────────────────────→ /imu/data（带姿态）
                                                              │
激光雷达 ───→ lidar 驱动 ───────────────────────────────→ /scan_raw
                                                              │
         ┌────────────────────────────────────────────────────┤
         │            EKF (robot_localization)                 │
         │   输入: /odom_raw + /imu/data + /odom_rf2o(可选)    │
         │   输出: /odom（融合后）                              │
         │   发布: odom → base_footprint TF                    │
         └──────────────────────┬─────────────────────────────┘
                                │
          ┌─────────────────────┼─────────────────────────────┐
          │                     │                             │
     【建图模式】           【导航模式】                        │
  SLAM Toolbox              map_server ← 加载 .yaml 地图      │
  输入: /scan_raw + /odom   AMCL ← /scan_raw + /map           │
  输出: /map                输出: map → odom TF                │
  发布: map → odom TF            │                             │
          │                      ▼                             │
          │              Nav2 planner (A*) ← 全局路径规划       │
          │                      │                             │
          │                      ▼                             │
          │              Nav2 controller (MPPI/DWB)            │
          │              输入: 路径 + /odom + costmap           │
          │                      │                             │
          │                      ▼                             │
          │               /cmd_vel_nav                         │
          │                      │                             │
          │              velocity_smoother                      │
          │                      │                             │
          └──────────────────────┼─────────────────────────────┘
                                 ▼
                          /cmd_vel → ros_robot_controller → 电机转动
```

### 0.5 文件速查表：每个配置文件管什么

| 文件路径 | 管什么 | 你需要改什么 |
|---------|--------|------------|
| `simulations/rosorin_description/urdf/mecanum.xacro` | 麦轮车 URDF 模型（尺寸、传感器位置） | 实测 base_link→lidar、imu 的 xyz 偏移 |
| `simulations/rosorin_description/urdf/acker.xacro` | 阿克曼车 URDF 模型 | 同上 |
| `driver/controller/config/calibrate_params.yaml` | 轮距、轮半径、编码器参数 | 实测你的底盘参数 |
| `driver/controller/config/ekf.yaml` | EKF 传感器融合配置 | 麦轮需开 vy；确认话题名匹配 |
| `slam/config/slam.yaml` | SLAM Toolbox 建图参数 | 确认 `scan_topic`、`base_frame` |
| `navigation/config/nav2_params.yaml` | Nav2 导航主配置（AMCL+costmap+planner+BT） | AMCL 粒子数、雷达量程、costmap 尺寸 |
| `navigation/config/nav2_controller_dwb.yaml` | DWB 局部控制器参数 | ⚠️ **需替换为 MPPI 配置** |
| `navigation/config/nav2_controller_teb.yaml` | TEB 局部控制器参数（备选） | 不用改（不用 TEB） |
| `peripherals/config/lidar_filters_config_*.yaml` | 各型号雷达滤波配置 | 选对你的型号即可 |

---

### 0.6 现有工程 vs 清单目标：差距在哪

| 清单模块 | 工程里已有 | 差距（需要你动手的） |
|---------|-----------|-------------------|
| ①URDF+TF | ✅ `mecanum.xacro` 已有完整 TF 树 | 实测改尺寸数值 |
| ②底盘驱动 | ✅ `ros_robot_controller` + `odom_publisher` | 确认 covariance 不全 0 |
| ③激光雷达 | ✅ `lidar.launch.py` 支持多型号 | 设对 `LIDAR_TYPE` 环境变量 |
| ④IMU | ✅ `imu_filter.launch.py` 已集成 | 确认 IMU 坐标系方向 |
| ⑤EKF 融合 | ✅ `ekf.yaml` 已有完整配置 | 麦轮需确认 vy 融合开关 |
| ⑥SLAM 建图 | ✅ `slam.launch.py` 已用 SLAM Toolbox | 基本可直接跑 |
| ⑦AMCL 定位 | ✅ `nav2_params.yaml` 已配好 | 微调雷达量程参数 |
| ⑧全局规划 | ✅ A* NavfnPlanner 已配好 | 无需改 |
| ⑨局部控制 | ⚠️ 目前是 **DWB**，清单要求 **MPPI** | **需新建 MPPI 配置文件 + 改 launch** |

### 0.7 分步行动计划（按这个顺序一步步来）

> ⚠️ **铁律**：每完成一步必须验证通过，再进入下一步。不要跳步。

---

#### 🔹 阶段一：让底盘能动、数据能出来（第 1~4 步）

| 步骤 | 做什么 | 改哪个文件 | 验证命令 | 通过标准 |
|:---:|--------|-----------|---------|---------|
| **1** | 设置环境变量 | `~/.bashrc` 或 launch 前 export | `echo $LIDAR_TYPE` | 输出你的雷达型号 |
| **2** | 实测底盘尺寸，改 URDF | `urdf/mecanum.xacro`（或 `acker.xacro`） | `ros2 run tf2_tools view_frames` | TF 树有 base_footprint→base_link→lidar_frame→imu_link，xyz 与实测一致 |
| **3** | 实测轮距轮半径，改标定参数 | `controller/config/calibrate_params.yaml` | 推车走 1 米，`ros2 topic echo /odom_raw` | 显示约 1.0 米位移 |
| **4** | 单独启动底盘+雷达验证 | 无需改文件 | `ros2 launch slam slam.launch.py` 的基座部分 | `/scan_raw` 有数据、`/odom_raw` 静止不漂 |

**第 1~4 步完成后，你有了**：✅ TF 树 + ✅ 轮速里程计 + ✅ 雷达点云 + ✅ IMU 数据

---

#### 🔹 阶段二：EKF 融合跑通（第 5~6 步）

| 步骤 | 做什么 | 改哪个文件 | 验证命令 | 通过标准 |
|:---:|--------|-----------|---------|---------|
| **5** | 确认 EKF 配置匹配你的话题 | `controller/config/ekf.yaml` | 检查 `odom0: odom_raw`、`imu0: imu` 是否对应实际话题名 | 话题名一一对应 |
| **6** | 启动 controller.launch.py | 无需改 | `ros2 topic echo /odom` | 有融合输出，推车时数据平滑；`ros2 run tf2_ros tf2_echo odom base_footprint` 有变换 |

**第 5~6 步完成后，你有了**：✅ 融合里程计 `/odom`（EKF 输出，给后面所有模块用）

---

#### 🔹 阶段三：SLAM 建图（第 7~8 步）

| 步骤 | 做什么 | 改哪个文件 | 验证命令 | 通过标准 |
|:---:|--------|-----------|---------|---------|
| **7** | 确认 slam.yaml 的 frame/topic 名 | `slam/config/slam.yaml` | 检查 `base_frame`、`scan_topic` 与实际一致 | 参数匹配 |
| **8** | 启动建图 + 遥控走一圈 + 存图 | 无需改 | RViz2 看地图；存图后检查 `.yaml` + `.pgm` | 地图无撕裂，墙壁清晰 |

**第 7~8 步完成后，你有了**：✅ 一张可用的栅格地图

---

#### 🔹 阶段四：导航跑通（第 9~13 步）

| 步骤 | 做什么 | 改哪个文件 | 验证命令 | 通过标准 |
|:---:|--------|-----------|---------|---------|
| **9** | 微调 AMCL 参数 | `navigation/config/nav2_params.yaml` | — | `laser_max_range` 匹配你的雷达量程 |
| **10** | 新建 MPPI 控制器配置 | **新建** `navigation/config/nav2_controller_mppi.yaml` | — | 文件内容参考清单第二节第8小节 |
| **11** | 改 launch 支持 MPPI | `navigation/launch/include/navigation_base.launch.py` | — | 加一个 `use_mppi` 分支加载新配置 |
| **12** | 启动导航全栈 | 无需改 | `ros2 launch navigation navigation.launch.py map:=你的地图名` | Nav2 所有 lifecycle 节点 active |
| **13** | RViz2 点目标测试 | — | 点 "2D Goal Pose" | 小车自动规划+行驶到目标点 |

**第 9~13 步完成后，你有了**：✅ 完整的自动导航能力

---

#### 🔹 阶段五：调优 + 视觉接入（第 14~15 步，进阶）

| 步骤 | 做什么 | 说明 |
|:---:|--------|------|
| **14** | MPPI 参数调优 | 阿克曼 `vy_max=0.0`，麦轮 `vy_max=0.5`；`batch_size` 从 2000 起调 |
| **15** | 接入偏振光视觉（清单第七节） | 按层次一→二→三逐步推进 |

## 一、整套最终标配（算法 + ROS2 包）

> **架构说明**：全局规划负责找路，局部控制器（MPPI）同时承担轨迹跟踪 + 避障，不需要再单独跑 Pure Pursuit。

| # | 模块 | 算法 / 包 | 状态 |
|:---:|------|----------|:---:|
| 1 | 机器人模型 & TF | URDF + `robot_state_publisher` | ⬜ |
| 2 | 底盘驱动 + 里程计 | 阿克曼 / 麦轮运动学 + PID | ✅ |
| 3 | 激光雷达驱动 | 按型号选包 | ⬜ |
| 4 | IMU 驱动 | 按型号选包 + `imu_filter_madgwick` | ⬜ |
| 5 | 传感器融合 | EKF（`robot_localization`）https://www.bilibili.com/video/BV1jK4y1U78V/?share_source=copy_web&vd_source=3e1e3ef48cb60ea8a57673546a95f085 | ⬜ |
| 6 | 建图 | 🔄 SLAM Toolbox（替换 Cartographer） | ⬜ |
| 7 | 定位 | AMCL（`nav2_amcl`）https://www.bilibili.com/video/BV1n54y157rS/?share_source=copy_web&vd_source=3e1e3ef48cb60ea8a57673546a95f085 | ⬜ |
| 8 | 全局规划 | A\*（`nav2_navfn_planner`） https://www.bilibili.com/video/BV1bv411y79P/?share_source=copy_web&vd_source=3e1e3ef48cb60ea8a57673546a95f085 | ⬜ |
| 9 | 局部控制 | 🔄 MPPI（替换 DWA + Pure Pursuit） | ⬜ |

## 二、各模块详细对应

### 1）机器人模型 & TF ⬜

> ⚠️ 必做，不然坐标系全歪。

| 项目 | 内容 |
|------|------|
| **作用** | 描述机器人尺寸、传感器位置、坐标系树 |
| **算法** | URDF + robot_state_publisher |
| **ROS2 包** | `robot_state_publisher`、`joint_state_publisher` |
| **核心话题** | `/tf`、`/tf_static` |

**必须写对的参数：**
- 阿克曼：轴距 `wheelbase`、轮距 `track_width`、最大转角 `max_steer_angle`
- 麦轮：四轮中心到几何中心的 x/y 距离、轮子半径
- 激光雷达 `laser`（或 `lidar_link`）相对 `base_link` 的 tf

**调试步骤：**
```bash
# 1. 静态验证 TF 树是否完整
ros2 run tf2_tools view_frames

# 2. 检查某两帧之间的变换
ros2 run tf2_ros tf2_echo base_link laser
```

---

### 2）底盘驱动 + 里程计 ✅

> 已有完善的阿克曼和麦轮驱动程序，以下为对接 ROS2 的要点。

| 项目 | 内容 |
|------|------|
| **作用** | 读编码器、发速度指令、发布 odom |
| **ROS2 节点** | 已有驱动节点 |
| **订阅话题** | `/cmd_vel`（geometry_msgs/Twist） |
| **发布话题** | `/odom`（nav_msgs/Odometry）、`/tf`（odom→base_link） |

**对接检查清单：**
- [ ] `/odom` 的 `pose.covariance` 和 `twist.covariance` 要填合理值（不能全填 0）
- [ ] odom → base_link 的 TF 必须由驱动节点发布（或 `robot_localization` 发布）
- [ ] 麦轮里程计需正确处理四轮联合运动学（`vx`、`vy`、`ω` 三个分量都要输出）
- [ ] 验证命令：静止小车，`ros2 topic echo /odom`，位置应不漂移

---

### 3）激光雷达驱动 ⬜

| 项目 | 内容 |
|------|------|
| **作用** | 输出激光点云 |
| **话题** | `/scan`（sensor_msgs/LaserScan） |

**常用包（按型号选）：**
- `ydlidar_ros2_driver`（亿达）
- `rplidar_ros`（思岚）
- `sllidar_ros2` 等

**验证：**
```bash
ros2 topic hz /scan   # 确认频率正常，一般 10~15 Hz
```

---

### 4）IMU 驱动 ⬜

| 项目 | 内容 |
|------|------|
| **作用** | 输出角速度、线加速度，辅助 EKF 修正漂移 |
| **ROS2 包** | 视具体型号，或串口自驱 + `imu_tools` |
| **话题** | `/imu/data`（sensor_msgs/Imu） |

**对接要点：**
- [ ] 确认 IMU 坐标系和 `base_link` 方向一致（或在 URDF 里写好 tf）
- [ ] 如果 IMU 自带姿态解算，发布 `orientation`；如果只有裸数据，用 `imu_filter_madgwick` 先融合
- [ ] 用 `imu_tools` 的 `imu_transformer` 做坐标系转换

---

### 5）传感器融合 EKF ⬜

> ⚠️ 实机必开，纯轮速里程计必飘。

| 项目 | 内容 |
|------|------|
| **作用** | 轮速里程计 + IMU → 稳定连续位姿估计 |
| **算法** | EKF 扩展卡尔曼滤波 |
| **ROS2 包** | `robot_localization` |
| **配置文件** | `ekf.yaml` |
| **输入** | `/odom`、`/imu/data` |
| **输出** | `/odometry/filtered`（给 Nav2 用）、发布 odom→base_link TF |

**`ekf.yaml` 关键配置说明：**
```yaml
ekf_filter_node:
  ros__parameters:
    frequency: 50.0          # 输出频率，建议 ≥ 控制频率
    sensor_timeout: 0.1

    odom0: /odom
    odom0_config: [true,  true,  false,   # x, y, z
                   false, false, true,    # roll, pitch, yaw
                   true,  true,  false,   # vx, vy, vz
                   false, false, true,    # vroll, vpitch, vyaw
                   false, false, false]   # ax, ay, az
    # 麦轮需开启 vy：odom0_config 第 8 位改为 true

    imu0: /imu/data
    imu0_config: [false, false, false,
                  true,  true,  true,    # 融合姿态角
                  false, false, false,
                  true,  true,  true,    # 融合角速度
                  true,  true,  false]   # 融合线加速度（z 轴一般不融合）

    publish_tf: true                     # 由 EKF 发布 odom→base_link
    world_frame: odom
```

---

### 6）建图 SLAM 🔄 SLAM Toolbox（替换 Cartographer）

> **为什么换？**  
> Cartographer 在 Jetson ARM 平台编译繁琐、依赖重，且 ROS2 Humble 社区维护活跃度下降。  
> **SLAM Toolbox** 是 Nav2 官方推荐的默认 SLAM 方案，支持在线建图、地图续建（lifelong mapping），ARM 上编译无障碍，且可直接复用已有地图定位（替代部分 AMCL 场景）。

| 项目 | 内容 |
|------|------|
| **ROS2 包** | `slam_toolbox` |
| **输入** | `/scan`、`/odometry/filtered`（或 `/odom`）、`/tf` |
| **输出** | `/map`、map→odom TF |

**启动建图：**
```bash
ros2 launch slam_toolbox online_async_launch.py \
  slam_params_file:=./slam_toolbox_params.yaml
```

**保存地图：**
```bash
# 方式1：service 调用（推荐，可保存 .posegraph 供续建）
ros2 service call /slam_toolbox/save_map slam_toolbox/srv/SaveMap \
  "{name: {data: 'my_map'}}"

# 方式2：兼容 nav2_map_server 格式
ros2 run nav2_map_server map_saver_cli -f my_map
```

---

### 7）定位 ⬜

| 项目 | 内容 |
|------|------|
| **算法** | AMCL 粒子滤波 |
| **ROS2 包** | `nav2_amcl` |
| **输入** | `/map`、`/scan`、`/odometry/filtered` |
| **输出** | map→odom TF，即全局定位结果 |

**关键参数（`nav2_params.yaml` 中）：**
```yaml
amcl:
  min_particles: 500        # 粒子太少定位跳，太多卡（Orin 可适当调大）
  max_particles: 3000
  laser_max_range: 8.0      # 匹配雷达实际量程
  laser_min_range: 0.15
  update_min_d: 0.1         # 平移多少更新一次粒子
  update_min_a: 0.1         # 旋转多少更新一次粒子
```

**初始位姿设置：**
```bash
# 方式1：RViz2 点击 "2D Pose Estimate"
# 方式2：代码发布
ros2 topic pub /initialpose geometry_msgs/PoseWithCovarianceStamped ...
```

---

### 8）导航栈 Nav2 ⬜

| 功能 | 算法 | ROS2 包 | 备注 |
|------|------|---------|------|
| 全局路径规划 | A\* | `nav2_navfn_planner` | — |
| 局部控制 & 避障 & 轨迹跟踪 | 🔄 **MPPI** | `nav2_mppi_controller` | 同时替换 DWA + Pure Pursuit |

> **为什么用 MPPI 同时替换 DWA 和 Pure Pursuit？**
> - **DWA**：在 ROS2 Humble 中已被官方标记为弃用，未来版本将移除
> - **Pure Pursuit**：纯几何算法，只追前视点，不具备避障能力，功能是 MPPI 的子集
> - **MPPI** 同时完成轨迹跟踪 + 实时避障，利用 Jetson Orin GPU 并行采样，一个控制器做两件事，不需要再叠加 Pure Pursuit

**MPPI 关键参数（`nav2_params.yaml`）：**
```yaml
controller_server:
  ros__parameters:
    controller_plugins: ["FollowPath"]
    FollowPath:
      plugin: "nav2_mppi_controller::MPPIController"
      time_steps: 56
      model_dt: 0.05
      batch_size: 2000        # Orin GPU 可适当调大到 4000~8000
      vx_max: 0.5
      vx_min: -0.1
      vy_max: 0.5             # 麦轮开启，阿克曼设为 0.0
      wz_max: 1.0
      iteration_count: 1
```

**完整 Nav2 包列表：**
```
nav2_bringup
nav2_planner
nav2_controller
nav2_recoveries
nav2_bt_navigator
nav2_map_server
nav2_amcl
```

## 三、实机标准启动顺序

> ⚠️ **严格按顺序逐层验证，不要一把全开。**

### 🗺️ 路线 1：建图模式

| 步骤 | 操作 | 验证方式 |
|:---:|------|---------|
| 1 | 启动 URDF + TF | `view_frames` 确认 TF 树完整 |
| 2 | 启动底盘驱动 ✅ | `echo /odom` 静止不漂移 |
| 3 | 启动激光雷达 | `hz /scan` 频率正常 |
| 4 | 启动 IMU | `echo /imu/data` 静止角速度接近 0 |
| 5 | 启动 EKF 融合 | `echo /odometry/filtered` 有输出 |
| 6 | 启动 SLAM Toolbox | RViz2 看到地图在扩展 |
| 7 | 手柄/键盘遥控走一圈 | 地图无明显跳变或撕裂 |
| 8 | 保存地图 | 确认 `.yaml` + `.pgm` 文件生成 |

---

### 🚗 路线 2：导航模式（全自动）

| 步骤 | 操作 | 验证方式 |
|:---:|------|---------|
| 1 | 启动 URDF + TF | TF 树完整 |
| 2 | 启动底盘驱动 ✅ | `/odom` 正常 |
| 3 | 启动激光雷达 | `/scan` 正常 |
| 4 | 启动 IMU | `/imu/data` 正常 |
| 5 | 启动 EKF 融合 | `/odometry/filtered` 正常 |
| 6 | 启动 map_server（加载地图） | RViz2 地图显示正确 |
| 7 | 启动 AMCL 定位 | RViz2 粒子云收敛到正确位置 |
| 8 | 启动 Nav2（MPPI 控制器） | `NavigateToPose` action server 上线 |
| 9 | RViz2 点 "2D Goal Pose" | 小车自动规划路径并行驶 |

## 四、launch 结构

```
bringup.launch.py
├── robot_state_publisher          # URDF → TF 静态变换
├── 底盘驱动节点 ✅                 # 发布 /odom，订阅 /cmd_vel
├── 激光雷达驱动                    # 发布 /scan
├── IMU 驱动                       # 发布 /imu/data
├── imu_filter_madgwick            # 仅 IMU 无姿态解算时需要
├── robot_localization (EKF)       # 融合输出 /odometry/filtered
├── slam_toolbox                   # 【建图模式】在线建图 + 回环检测
├── map_server                     # 【导航模式】加载已有地图
├── nav2_amcl                      # 【导航模式】粒子滤波全局定位
└── nav2_bringup
    ├── planner_server (A*)        # 全局路径规划
    └── controller_server (MPPI)   # 轨迹跟踪 + 局部避障（二合一）
```

**Jetson Orin 上的环境准备：**
```bash
# 确认 JetPack 版本（需 5.1+ 以支持 Ubuntu 22.04 + ROS2 Humble）
cat /etc/nv_tegra_release

# 安装 Nav2 全家桶
sudo apt install ros-humble-navigation2 ros-humble-nav2-bringup

# 安装 SLAM Toolbox
sudo apt install ros-humble-slam-toolbox

# 安装 robot_localization
sudo apt install ros-humble-robot-localization

# 安装 MPPI（包含在 nav2 中，Humble 需确认版本）
sudo apt install ros-humble-nav2-mppi-controller
```

## 五、实机最容易炸的参数

| # | 参数 / 配置 | 后果 | 针对你的平台 |
|:---:|------------|------|------------|
| 1 | **TF 树完整性** `base_link→laser`、`odom→base_link→map` | 定位/建图全歪 | URDF 里激光和轮子位置要实测 |
| 2 | **EKF `publish_tf` 与驱动节点冲突** | odom TF 重复发布导致抖动 | 驱动节点关掉 TF 发布，交给 EKF |
| 3 | **麦轮里程计 `vy` 分量** | EKF 融合错误，小车横移时估计崩掉 | `odom0_config` 第 8 位必须为 true |
| 4 | **AMCL 粒子数** `min/max_particles` | 太少定位跳；太多 CPU 卡 | Orin 可设 500~3000，够用 |
| 5 | **MPPI `batch_size`** | 太小避障差；太大 GPU 显存溢出 | 从 2000 开始，Orin 8G 可试 4000 |
| 6 | **底盘尺寸**（轴距、轮距、轮半径） | 导航路径规划错误，必撞 | 实测，误差 < 5mm |
| 7 | **传感器时间戳同步** | EKF 融合延迟，定位抖动 | 驱动节点用 `rclcpp::Clock` 统一时间源 |

## 六、待办事项

- [x] ~~写完/调好底盘驱动 + 里程计~~（阿克曼 & 麦轮驱动已完善 ✅）
- [ ] 完成 URDF 建模，验证 TF 树
- [ ] 跑通 EKF 融合（`/odometry/filtered` 稳定输出）
- [ ] 用 SLAM Toolbox 建一张干净地图
- [ ] 启动 Nav2 + AMCL + MPPI，实现全自动导航
- [ ] 针对阿克曼/麦轮分别调 MPPI 参数（`vy_max` 区别对待）

---

> 💬 下一步可以提供：**`ekf.yaml` 完整模板**、**`nav2_params.yaml` 针对阿克曼/麦轮的双配置模板**，或 **SLAM Toolbox 建图 launch 文件**，按需告知。

## 七、视觉模块接入规划（偏振光视觉）

> 按难度/收益分三个层次，建议按顺序推进。

---

### 层次一：视觉感知输出，不碰导航核心 ✅ 最容易落地

视觉模块独立运行，识别结果发布为 ROS2 话题，导航栈完全不动。

```
相机驱动  →  /camera/image_raw
                └→ 偏振光处理节点（实验室算法）
                       └→ /detection/objects (vision_msgs/Detection2DArray)
                                └→ 上层任务节点（触发停止 / 转向 / 执行任务）
```

**适用场景**：识别特定目标后触发行为，路径规划仍由 Nav2 负责。

---

### 层次二：视觉里程计接入 EKF ⭐ 毕设核心亮点

把偏振光视觉里程计作为第三路传感器输入 EKF，与轮速计 + IMU 三路融合。

```
偏振光相机
  └→ 视觉里程计节点（ORB-SLAM3 / 自研）
       └→ /vo/odometry (nav_msgs/Odometry)
            └→ robot_localization EKF
                 ├─ /odom         ← 轮速计
                 ├─ /imu/data     ← IMU
                 └─ /vo/odometry  ← 偏振光视觉（新增）
```

**论文亮点句**：在强反光 / 逆光 / 高光污染场景下，引入偏振光视觉里程计后定位误差较轮速 + IMU 基线降低 XX%。

---

### 层次三：偏振光语义地图 + MPPI 代价函数融合 🔬 SCI 方向

在 SLAM Toolbox 几何地图基础上叠加偏振光语义层，并将语义信息注入导航代价函数。

```
SLAM Toolbox       →  几何栅格地图
偏振光视觉检测     →  语义标注（材质 / 危险区 / 目标区）
                            ↓
                   语义栅格地图 /semantic_costmap
                            ↓
                   MPPI 自定义 cost function
                   （危险区权重↑，安全材质区权重↓）
```

**关键模块实现：**
- 偏振光材质分类网络（输出每像素语义标签）
- 语义标签 → 栅格代价映射节点
- Nav2 `CostmapLayer` 插件（将语义代价叠加到 costmap）
- MPPI cost function 中增加语义代价项

---

### 关于层次三能否发 SCI

**单靠"偏振光 + 语义地图"本身不够，但加上下面任意一个就有机会：**

| 补强方向 | 说明 | 目标期刊档次 |
|----------|------|:---:|
| 偏振光材质分类的**网络设计**有创新（轻量化/新架构） | 视觉算法本身是创新点 | Q2~Q3 |
| 系统性对比实验：普通相机 vs 偏振光，多场景定量评估 | 数据驱动型论文 | Q3 |
| 偏振光辅助的**语义 SLAM**（建图+定位+语义三合一） | 系统级创新 | Q1~Q2 |
| 偏振光 + 事件相机 / 深度相机**多模态融合** | 传感器融合创新 | Q1 |

**直接建议**：

> 层次三本身是一个**系统集成工作**，如果偏振光检测算法（你们实验室的独门绝技）在准确率上有可量化的显著提升，核心创新点就在算法本身，系统集成是验证平台。  
> 这个组合——**偏振光检测算法（创新）+ 移动机器人语义导航（验证平台）**——投 *IEEE Transactions on Intelligent Transportation Systems* / *Robotics and Autonomous Systems* / *IEEE RA-L* 一类期刊是合理的，关键看实验数据是否扎实。

**发 SCI 最重要的不是系统多复杂，而是：**
1. 对比基线要够强（普通 RGB 相机 + 同等算法）
2. 场景要有代表性（强光、水面反射、玻璃地面等偏振光有优势的场景）
3. 指标要全（准确率、定位误差、导航成功率、实时性）

---

## 零·补充：完整 launch 调用树（从入口到最底层代码）

> 建图模式的完整调用链，每一层都标出对应的真实文件路径。

```
【入口】
slam/launch/slam.launch.py
│
├─ slam/launch/include/robot.launch.py          ← 基座：底盘+传感器+EKF
│  │
│  ├─ driver/controller/launch/controller.launch.py
│  │  │
│  │  ├─ driver/controller/launch/odom_publisher.launch.py
│  │  │  │
│  │  │  ├─ simulations/rosorin_description/launch/robot_description.launch.py
│  │  │  │  │  读取: urdf/rosorin.xacro（主 xacro，include mecanum/acker）
│  │  │  │  ├─ Node: robot_state_publisher   ← 解析 xacro，发布静态 TF
│  │  │  │  └─ Node: joint_state_publisher   ← 发布关节状态
│  │  │  │
│  │  │  ├─ driver/ros_robot_controller/launch/ros_robot_controller.launch.py
│  │  │  │  └─ Node: ros_robot_controller    ← 【最底层】读编码器/IMU，控电机
│  │  │  │       源码: ros_robot_controller/ros_robot_controller_node.py
│  │  │  │             ros_robot_controller/ros_robot_controller_sdk.py
│  │  │  │       发布: /ros_robot_controller/imu_raw, 订阅: /cmd_vel
│  │  │  │
│  │  │  └─ Node: odom_publisher             ← 【最底层】轮速→里程计
│  │  │       源码: controller/odom_publisher_node.py
│  │  │             controller/mecanum.py（或 ackermann.py）
│  │  │       配置: controller/config/calibrate_params.yaml
│  │  │       发布: /odom_raw
│  │  │
│  │  ├─ peripherals/launch/imu_filter.launch.py
│  │  │  ├─ Node: imu_calib (apply_calib)    ← IMU 零偏校正
│  │  │  │       配置: calibration/config/imu_calib.yaml
│  │  │  │       订阅: /ros_robot_controller/imu_raw → 发布: imu_corrected
│  │  │  └─ Node: imu_filter (complementary_filter_node)  ← 互补滤波，补全姿态
│  │  │       订阅: imu_corrected → 发布: /imu（带 orientation）
│  │  │
│  │  ├─ Node: ekf_filter_node               ← EKF 传感器融合
│  │  │       包: robot_localization
│  │  │       配置: controller/config/ekf.yaml
│  │  │       订阅: /odom_raw + /imu
│  │  │       发布: /odom（融合后）+ odom→base_footprint TF
│  │  │
│  │  └─ driver/servo_controller/launch/servo_controller.launch.py
│  │       └─ Node: servo_controller         ← 舵机控制（云台，非导航核心）
│  │
│  ├─ peripherals/launch/lidar.launch.py
│  │  │  读取环境变量 LIDAR_TYPE 选择驱动
│  │  └─ peripherals/launch/include/ldlidar_LD19.launch.py（以 LD19 为例）
│  │       └─ Node: ldlidar_stl_ros2_node    ← 【最底层】雷达驱动
│  │            配置: port_name=/dev/lidar, frame_id=lidar_frame
│  │            发布: /scan_raw
│  │
│  └─ peripherals/launch/joystick_control.launch.py
│       └─ Node: joystick_control            ← 手柄遥控节点
│            源码: peripherals/joystick_control.py
│            发布: /controller/cmd_vel
│
└─ slam/launch/include/slam_base.launch.py   ← 延迟 10s 启动
    │  读取配置: slam/config/slam.yaml（RewrittenYaml 动态替换帧名）
    └─ Node: sync_slam_toolbox_node          ← 【最底层】SLAM Toolbox 建图
         包: slam_toolbox
         订阅: /scan_raw + /odom + /tf
         发布: /map + map→odom TF
```

**最底层真正执行逻辑的代码文件**（不再调用任何 launch）：

| 文件 | 职责 |
|------|------|
| `ros_robot_controller/ros_robot_controller_node.py` | 读编码器、读 IMU 原始数据、控制电机 |
| `controller/odom_publisher_node.py` + `mecanum.py` | 轮速运动学→计算里程计 |
| `slam_toolbox`（系统包） | SLAM 建图算法本体 |
| `robot_localization`（系统包） | EKF 融合算法本体 |
| `imu_complementary_filter`（系统包） | IMU 互补滤波 |
| `ldlidar_stl_ros2_node`（系统包） | 雷达驱动本体 |